# 04 · Next facet suggestion: using the interaction history

In **next facet suggestion** (paper §4.3), a scenario's target facets are revealed one at a time in a fixed order. At step *t* the method receives `history`, the *t* facets already placed (oldest first), and suggests what should come next. Each step is costed separately, and a scenario's cost is the sum over its steps. Retrieval metrics are computed against the targets not yet placed.

The toy method below uses the history: it suggests unplaced facets from the apps already in use, nearest in the facet graph first, and fills the rest with globally popular facets.

In [ ]:
from collections import Counter
from contextxr.methods import Suggester


class SameAppNext(Suggester):
    name = "same_app_next (ours)"

    def fit(self, train):
        self.popular = Counter(w.facet_id for s in train for w in s.widgets if w.facet_id)

    def suggest(self, context, catalog, history=None, k=10):
        placed = set(history or [])
        near = {}
        for p in placed:
            graph = catalog.graph_of(p)
            for f in graph.nodes:
                hops = graph.hops(p, f)
                if f not in placed and hops is not None:
                    near[f] = min(near.get(f, hops), hops)
        ranked = sorted(near, key=lambda f: (near[f], -self.popular[f], f))
        ranked += [f for f, _ in self.popular.most_common() if f not in placed and f not in near]
        return ranked[:k]

In [ ]:
from IPython.display import Markdown
from contextxr import evaluate, load_minexrpp
from contextxr.methods import GlobalPopularity
from contextxr.runner import format_table

ds = load_minexrpp()
ours = evaluate(SameAppNext(), "next", ds)
popularity = evaluate(GlobalPopularity(), "next", ds)
Markdown(format_table([ours, popularity]))

| method | n | manual | cost | reduction | Recall@10 | F1@10 | Hit@10 |
|---|---:|---:|---:|---:|---:|---:|---:|
| same_app_next (ours) | 109 | 21.19 | 15.86 | -25.2% | 0.322 | 0.150 | 0.650 |
| popularity | 109 | 21.19 | 18.70 | -11.8% | 0.168 | 0.089 | 0.466 |

## Step by step for one scenario

In [ ]:
record = next(r for r in ours.records if r["scenario_id"] == "P04_LivingRoom_Relaxing")
for step in record["steps"]:
    t = step["hsm"]["targets"][0]
    print(f"t={step['t']}  history={len(step['history'])}  target={step['target']:32} "
          f"{t['outcome']}  cost {step['cost']['suggest']:.1f} vs manual {step['cost']['manual']:.1f}")

Task options such as the step order, or what happens when a method cannot suggest, are documented in `contextxr/tasks/next_suggestion.py`. The defaults reproduce the paper.